<a href="https://colab.research.google.com/github/Shauryasawant/footpath-damage-segmentation/blob/main/footpath_damage_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Footpath Damage Segmentation

A U-Net (EfficientNet-B1 encoder) that segments footpath damage (cracks, potholes, missing pavers) from a photo or video, scores severity, and optionally geotags each detection.

This notebook only covers **using the already-trained model**: load the checkpoint, run inference on your own images/video, and (optionally) evaluate it on the labeled test set. Training and fine-tuning code lives in the separate `training/` notebook / scripts in this repo, so this notebook stays fast and simple for anyone who just wants predictions.

**To use this notebook:**
1. Download `best_model.pt` from the [Releases page](checkpoints) of this repo (see the README for why it's not committed directly).
2. Put it at `checkpoints/best_model.pt` relative to this notebook (or update `CKPT_PATH` below).
3. Run all cells top to bottom.


## 1. Setup

In [ ]:
!pip install -q segmentation_models_pytorch albumentations

import os
import numpy as np
import cv2
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from scipy.ndimage import binary_closing, binary_opening, label as nd_label
from skimage import morphology

# Running in Colab: mount Drive if your checkpoint/data live there.
# Running locally / on a fresh clone: this is skipped automatically.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 2. Load the trained model

Points at `checkpoints/best_model.pt` by default. If you're running this in Colab against your own Drive copy instead of a local clone, change `CKPT_PATH` to your Drive path.

In [ ]:
CKPT_PATH = "checkpoints/best_model.pt"   # <-- change if your checkpoint lives elsewhere
ENCODER_NAME = "efficientnet-b1"
IMG_SIZE = 512

assert os.path.exists(CKPT_PATH), (
    f"Checkpoint not found at {CKPT_PATH}. Download best_model.pt from the repo's "
    f"Releases page and place it there, or update CKPT_PATH above."
)

model = smp.Unet(
    encoder_name=ENCODER_NAME,
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

dice_shown = ckpt.get("val_dice", ckpt.get("orig_val_dice"))
print(f"Loaded checkpoint (val_dice={dice_shown:.4f})" if dice_shown is not None else "Loaded checkpoint")

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])


## 3. Severity scoring and inference helpers

Severity is a weighted combination of:
- **Coverage** (fraction of the frame that's damaged) — weighted higher, since it's what actually separates a small crack from a large pothole.
- **Medial-axis width** — catches linear damage (cracks) that coverage alone would underrate.


In [ ]:
SEVERITY_COVERAGE_REF = 0.5       # coverage_frac that saturates the coverage component
SEVERITY_WIDTH_REF_PX = 15.0      # medial-axis width that saturates the width component
SEVERITY_COVERAGE_WEIGHT = 0.6    # vs (1 - this) for width
SEVERITY_SCORE_THRESHOLDS = (0.25, 0.55)  # low < 0.25, medium 0.25-0.55, high > 0.55


def score_severity(binary_mask: np.ndarray) -> dict:
    mask_u8 = binary_mask.astype(np.uint8)
    opened = binary_opening(binary_closing(mask_u8))
    labeled, num_components = nd_label(opened)
    coverage_frac = float(mask_u8.mean())

    if num_components == 0:
        return {"severity": "no_crack", "severity_score": 0.0, "max_width_px": 0.0,
                "total_length_px": 0.0, "coverage_frac": coverage_frac, "num_components": 0}

    total_length = 0.0
    max_width = 0.0
    for lbl in range(1, num_components + 1):
        component = labeled == lbl
        medial_axis, dist = morphology.medial_axis(component, return_distance=True)
        total_length += medial_axis.sum()
        if dist.size:
            max_width = max(max_width, dist.max())

    coverage_component = min(coverage_frac / SEVERITY_COVERAGE_REF, 1.0)
    width_component = min(max_width / SEVERITY_WIDTH_REF_PX, 1.0)
    w = SEVERITY_COVERAGE_WEIGHT
    severity_score = w * coverage_component + (1 - w) * width_component

    low_thr, high_thr = SEVERITY_SCORE_THRESHOLDS
    severity = "low" if severity_score < low_thr else ("medium" if severity_score < high_thr else "high")

    return {
        "severity": severity, "severity_score": float(severity_score),
        "max_width_px": float(max_width), "total_length_px": float(total_length),
        "coverage_frac": coverage_frac, "num_components": int(num_components),
    }


def predict(image_rgb: np.ndarray) -> dict:
    augmented = val_tf(image=image_rgb, mask=np.zeros(image_rgb.shape[:2], dtype="float32"))
    tensor = augmented["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor))[0, 0].cpu().numpy()
    binary_mask = (pred > 0.5).astype(np.uint8)
    result = score_severity(binary_mask)
    result["raw_pred"] = pred
    result["binary_mask"] = binary_mask
    return result


def show_result(image_rgb: np.ndarray, result: dict, title: str = ""):
    resized_img = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(resized_img); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(result["binary_mask"], cmap="gray"); axes[1].set_title("Predicted mask"); axes[1].axis("off")
    axes[2].imshow(resized_img); axes[2].imshow(result["raw_pred"], cmap="jet", alpha=0.45)
    axes[2].set_title(f"severity={result['severity']} ({result['severity_score']:.2f}) | coverage={result['coverage_frac']*100:.1f}%")
    axes[2].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## 4. Run inference on your own photo(s)

In Colab this opens a file picker. Outside Colab, drop images into an `images/` folder next to this notebook and it'll run over those instead.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    image_paths = list(uploaded.keys())
else:
    image_dir = "images"
    image_paths = [os.path.join(image_dir, f) for f in sorted(os.listdir(image_dir))] if os.path.isdir(image_dir) else []
    if not image_paths:
        print(f"No images found in ./{image_dir}/ -- add some .jpg/.png files there and rerun this cell.")

for path in image_paths:
    image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    result = predict(image)
    show_result(image, result, title=os.path.basename(path))
    print(f"{os.path.basename(path)}: severity={result['severity']} ({result['severity_score']:.2f}), "
          f"coverage={result['coverage_frac']*100:.1f}%")


## 5. Run inference on a video

Produces an annotated video with a live metadata panel (severity, coverage, location, accessibility) burned into each frame, plus a structured `issues.json` with one record per detected issue.

`pin_lat` / `pin_lon` is a manual location pin for the whole clip (no GPS wired up here) -- every record is honestly labeled `location.source = "manual_pin_fallback"` rather than pretending to be real GPS. Swap in the EXIF/GPX-based geotagging in `geotag_pipeline.py` (also in this repo) if you have real location data per frame.

In [ ]:
import json
import uuid
import datetime
import time


def classify_accessibility(coverage_frac):
    if coverage_frac < 0.02:
        return "CLEAR"
    elif coverage_frac < 0.10:
        return "PARTIALLY_BLOCKED"
    else:
        return "SEVERELY_BLOCKED"


def classify_issue_type(coverage_frac, max_width_px):
    types = []
    if coverage_frac > 0.15:
        types.append("broken_pavement")
    if 0.02 < coverage_frac <= 0.15:
        types.append("missing_tiles")
    if max_width_px > 12 and coverage_frac < 0.1:
        types.append("crack")
    if not types and coverage_frac > 0.005:
        types.append("surface_irregularity")
    return types


def estimate_confidence(severity_score, coverage_frac):
    """Heuristic, not a calibrated probability -- swap for real model
    confidence (e.g. mean predicted-mask probability in the damaged
    region) once that's plumbed through predict()."""
    base = 0.5 + 0.4 * min(severity_score, 1.0)
    if coverage_frac < 0.005:
        base *= 0.7
    return round(min(base, 0.98), 2)


def draw_hud(frame_bgr, mask, result, pin_lat, pin_lon, frame_idx, poi_id):
    h, w = frame_bgr.shape[:2]
    out = frame_bgr.copy()

    mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    red_layer = np.zeros_like(out)
    red_layer[mask_resized > 0] = (0, 0, 255)
    out = cv2.addWeighted(out, 1.0, red_layer, 0.35, 0)

    panel_h, panel_w = 175, 400
    panel_bg = out[10:10 + panel_h, 10:10 + panel_w].copy()
    solid = np.full_like(panel_bg, (20, 20, 20))
    out[10:10 + panel_h, 10:10 + panel_w] = cv2.addWeighted(panel_bg, 0.35, solid, 0.65, 0)

    severity_color = {
        "low": (0, 200, 0), "medium": (0, 165, 255), "high": (0, 0, 255), "no_crack": (0, 200, 0)
    }.get(result["severity"], (255, 255, 255))

    lines = [
        (f"POI: {poi_id}", (255, 255, 255)),
        (f"Severity: {result['severity'].upper()}  ({result['severity_score']:.2f})", severity_color),
        (f"Coverage: {result['coverage_frac']*100:.1f}%", (255, 255, 255)),
        (f"Access: {classify_accessibility(result['coverage_frac'])}", (255, 255, 255)),
        (f"Location: {pin_lat:.5f}, {pin_lon:.5f}", (180, 180, 180)),
        ("(manual pin -- GPS not yet wired)", (140, 140, 140)),
        (f"Frame: {frame_idx}", (150, 150, 150)),
    ]
    y = 35
    for text, color in lines:
        cv2.putText(out, text, (22, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 1, cv2.LINE_AA)
        y += 23

    return out


def process_video(video_path, output_video_path, output_json_path,
                   pin_lat, pin_lon, process_every_n_frames=10, min_coverage_to_log=0.005):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))

    records = []
    frame_idx = 0
    poi_counter = 0
    last_annotated = None
    t0 = time.time()

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        if frame_idx % process_every_n_frames == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            result = predict(frame_rgb)
            mask = (result["raw_pred"] > 0.5).astype(np.uint8) * 255

            poi_id = f"FP_{poi_counter:05d}"
            annotated = draw_hud(frame_bgr, mask, result, pin_lat, pin_lon, frame_idx, poi_id)
            last_annotated = annotated

            if result["coverage_frac"] >= min_coverage_to_log:
                record = {
                    "issue_id": f"fp_{datetime.datetime.now().strftime('%Y%m%d')}_{uuid.uuid4().hex[:8]}",
                    "timestamp": datetime.datetime.now().isoformat(),
                    "source": {"type": "video_frame", "file": video_path.split("/")[-1],
                               "frame_index": frame_idx, "frame_time_sec": round(frame_idx / fps, 2)},
                    "location": {"latitude": pin_lat, "longitude": pin_lon,
                                 "source": "manual_pin_fallback", "accuracy_m": None},
                    "footpath_detected": True,
                    "damage": {
                        "detected": result["severity"] != "no_crack",
                        "coverage_frac": round(result["coverage_frac"], 4),
                        "max_width_px": round(result["max_width_px"], 2),
                        "severity_score": round(result["severity_score"], 3),
                        "severity_label": result["severity"],
                    },
                    "issue_type": classify_issue_type(result["coverage_frac"], result["max_width_px"]),
                    "review_status": "unvalidated",
                    "confidence": estimate_confidence(result["severity_score"], result["coverage_frac"]),
                }
                records.append(record)
                poi_counter += 1
        else:
            # write through the last annotated frame's HUD so the panel doesn't
            # flicker blank between processed frames -- keeps playback smooth
            annotated = last_annotated if last_annotated is not None else frame_bgr

        writer.write(annotated)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx}/{total_frames} frames ({time.time()-t0:.0f}s elapsed)")

    cap.release()
    writer.release()

    with open(output_json_path, "w") as f:
        json.dump(records, f, indent=2)

    print(f"\nDone. Annotated video: {output_video_path}")
    print(f"Detection log ({len(records)} points): {output_json_path}")
    return records


### Upload and run

Update `pin_lat` / `pin_lon` to the real street this was filmed on.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded_video = files.upload()
    video_path = next(iter(uploaded_video))
else:
    video_path = "video.mp4"  # <-- put your video next to this notebook, or set the full path

records = process_video(
    video_path=video_path,
    output_video_path="annotated_output.mp4",
    output_json_path="issues.json",
    pin_lat=19.0760,   # <-- change to the real location
    pin_lon=72.8777,
    process_every_n_frames=10,  # raise this if on CPU and it's too slow
)


In [ ]:
# Re-encode to H.264 so it plays inline, then display it
!ffmpeg -y -i annotated_output.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 23 annotated_output_h264.mp4

from IPython.display import Video
Video("annotated_output_h264.mp4", embed=True, width=480)


---
## 5b. Geotagging with real location data

The video demo above uses a single manually-typed `pin_lat`/`pin_lon` for the whole clip -- fine for a quick demo, but not real per-detection location. This section extracts **actual GPS** wherever it's available, and produces one structured JSON record per detected issue with a proper metadata schema:

```
{
  "issue_id": ..., "timestamp": ..., "source": {...}, "location": {...},
  "footpath_detected": bool, "damage": {...}, "issue_type": [...],
  "review_status": "unvalidated", "confidence": float
}
```

Three ways a record's location gets filled in, tried in this order depending on what input you give it:

- **Path A -- photo EXIF GPS.** If a photo was taken with phone location on, its lat/lon/timestamp are embedded in the file and get read directly. Works per-photo, no extra setup.
- **Path B -- video + GPX log.** If you have a GPX track (exported from a phone GPS logger app, Strava, Google Maps Timeline, etc.) recorded alongside the video, each frame's timestamp is matched against the track by linear interpolation between the two nearest GPS fixes.
- **Fallback -- manual pin.** If neither exists for a given clip, you supply one lat/lon for the whole video by hand. Every such record is honestly labeled `location.source = "manual_pin_fallback"` rather than disguised as real GPS.

Install the extra dependencies this needs:

In [ ]:
!pip install -q exifread gpxpy


In [ ]:
import os
import json
import uuid
import datetime
import bisect

import cv2
import numpy as np
import exifread


# ============================================================================
# PATH A: EXIF GPS extraction from photos
# ============================================================================

def _dms_to_decimal(dms, ref):
    degrees = float(dms.values[0].num) / float(dms.values[0].den)
    minutes = float(dms.values[1].num) / float(dms.values[1].den)
    seconds = float(dms.values[2].num) / float(dms.values[2].den)
    decimal = degrees + minutes / 60.0 + seconds / 3600.0
    if ref in ("S", "W"):
        decimal = -decimal
    return decimal


def extract_exif_gps(image_path: str):
    """Returns (lat, lon, timestamp_iso, accuracy_m_or_None) or None if the
    photo has no embedded GPS tags. accuracy_m is None because EXIF rarely
    includes a GPS accuracy/HDOP field -- report it as unknown, don't guess."""
    with open(image_path, "rb") as f:
        tags = exifread.process_file(f, details=False)

    lat_tag = tags.get("GPS GPSLatitude")
    lon_tag = tags.get("GPS GPSLongitude")
    lat_ref = tags.get("GPS GPSLatitudeRef")
    lon_ref = tags.get("GPS GPSLongitudeRef")

    if not (lat_tag and lon_tag and lat_ref and lon_ref):
        return None

    lat = _dms_to_decimal(lat_tag, str(lat_ref))
    lon = _dms_to_decimal(lon_tag, str(lon_ref))

    dt_tag = tags.get("EXIF DateTimeOriginal") or tags.get("Image DateTime")
    if dt_tag:
        try:
            dt = datetime.datetime.strptime(str(dt_tag), "%Y:%m:%d %H:%M:%S")
            timestamp_iso = dt.isoformat()
        except ValueError:
            timestamp_iso = None
    else:
        timestamp_iso = None

    return lat, lon, timestamp_iso, None


# ============================================================================
# PATH B: GPS log (GPX) interpolation against video frame time
# ============================================================================

class GpsTrack:
    """Loads a GPX file (standard export from GPS Logger, Strava, Google
    Maps Timeline export, etc.) and looks up an interpolated lat/lon for
    any elapsed-seconds-since-video-start value.

    IMPORTANT: the GPS log's clock and the video's start time must be
    roughly synced. Simplest reliable method: note the wall-clock time you
    pressed record and pass it as video_start_time -- get this from your
    phone's video file metadata (creation time) or note it by hand when
    filming."""

    def __init__(self, gpx_path: str, video_start_time: datetime.datetime):
        import gpxpy
        with open(gpx_path, "r") as f:
            gpx = gpxpy.parse(f)

        points = []
        for track in gpx.tracks:
            for segment in track.segments:
                for point in segment.points:
                    points.append((point.time, point.latitude, point.longitude,
                                    getattr(point, "horizontal_dilution", None)))
        points.sort(key=lambda p: p[0])
        if not points:
            raise ValueError(f"No trackpoints found in {gpx_path}")

        self.video_start_time = video_start_time
        self.offsets = [(p[0] - video_start_time).total_seconds() for p in points]
        self.lats = [p[1] for p in points]
        self.lons = [p[2] for p in points]
        self.hdops = [p[3] for p in points]

    def lookup(self, elapsed_sec: float):
        """Linear interpolation between the two nearest GPS fixes.
        Returns (lat, lon, accuracy_m_estimate_or_None)."""
        idx = bisect.bisect_left(self.offsets, elapsed_sec)

        if idx == 0:
            return self.lats[0], self.lons[0], self._hdop_to_m(self.hdops[0])
        if idx >= len(self.offsets):
            return self.lats[-1], self.lons[-1], self._hdop_to_m(self.hdops[-1])

        t0, t1 = self.offsets[idx - 1], self.offsets[idx]
        frac = 0.0 if t1 == t0 else (elapsed_sec - t0) / (t1 - t0)
        lat = self.lats[idx - 1] + frac * (self.lats[idx] - self.lats[idx - 1])
        lon = self.lons[idx - 1] + frac * (self.lons[idx] - self.lons[idx - 1])
        return lat, lon, self._hdop_to_m(self.hdops[idx - 1])

    @staticmethod
    def _hdop_to_m(hdop):
        # rough rule of thumb: consumer GPS accuracy_m ~= HDOP * 5
        return float(hdop) * 5.0 if hdop is not None else None


# ============================================================================
# Issue classification (coarse, rule-based) and record schema
# ============================================================================

def classify_issue_type(coverage_frac: float, max_width_px: float, num_components: int) -> list:
    types = []
    if coverage_frac > 0.15:
        types.append("broken_pavement")
    if 0.02 < coverage_frac <= 0.15 and num_components > 2:
        types.append("missing_tiles")
    if max_width_px > 12 and coverage_frac < 0.1:
        types.append("crack")
    if not types and coverage_frac > 0.005:
        types.append("surface_irregularity")
    return types


def estimate_confidence(severity_score: float, coverage_frac: float) -> float:
    """Heuristic, not a calibrated probability -- swap for real model
    confidence (e.g. mean predicted-mask probability in the damaged
    region) once that's plumbed through predict()."""
    base = 0.5 + 0.4 * min(severity_score, 1.0)
    if coverage_frac < 0.005:
        base *= 0.7  # very small detections are less trustworthy
    return round(min(base, 0.98), 2)


def build_issue_record(result: dict, source: dict, location: dict) -> dict:
    num_components = result.get("num_components", 1)
    issue_types = classify_issue_type(result["coverage_frac"], result["max_width_px"], num_components)
    confidence = estimate_confidence(result["severity_score"], result["coverage_frac"])

    return {
        "issue_id": f"fp_{datetime.datetime.now().strftime('%Y%m%d')}_{uuid.uuid4().hex[:8]}",
        "timestamp": location.get("timestamp") or datetime.datetime.now().isoformat(),
        "source": source,
        "location": {
            "latitude": round(location["latitude"], 6),
            "longitude": round(location["longitude"], 6),
            "source": location["source"],
            "accuracy_m": location.get("accuracy_m"),
        },
        "footpath_detected": True,
        "damage": {
            "detected": result["severity"] != "no_crack",
            "coverage_frac": round(result["coverage_frac"], 4),
            "max_width_px": round(result["max_width_px"], 2),
            "severity_score": round(result["severity_score"], 3),
            "severity_label": result["severity"],
        },
        "issue_type": issue_types,
        "review_status": "unvalidated",
        "confidence": confidence,
    }


# ============================================================================
# Path A driver: process a folder of photos with EXIF GPS
# ============================================================================

def process_photos_with_exif(photo_paths: list, min_coverage_to_log: float = 0.005) -> list:
    records = []
    skipped_no_gps = []

    for path in photo_paths:
        gps = extract_exif_gps(path)
        if gps is None:
            skipped_no_gps.append(path)
            continue
        lat, lon, timestamp_iso, accuracy_m = gps

        image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        result = predict(image)  # from the inference helpers above

        if result["coverage_frac"] < min_coverage_to_log:
            continue

        record = build_issue_record(
            result,
            source={"type": "photo", "file": os.path.basename(path)},
            location={"latitude": lat, "longitude": lon, "source": "exif_gps",
                      "accuracy_m": accuracy_m, "timestamp": timestamp_iso},
        )
        records.append(record)

    if skipped_no_gps:
        print(f"WARNING: {len(skipped_no_gps)} photo(s) had no EXIF GPS and were skipped:")
        for p in skipped_no_gps:
            print(f"  {p}")
        print("(location was off, or the photo app/OS stripped EXIF on save/share -- "
              "check the original camera roll file, not a re-saved/compressed copy)")

    return records


# ============================================================================
# Path B driver: process video against a GPX log
# ============================================================================

def process_video_with_gps_log(video_path: str, gpx_path: str, video_start_time: datetime.datetime,
                                process_every_n_frames: int = 5, min_coverage_to_log: float = 0.005) -> list:
    track = GpsTrack(gpx_path, video_start_time)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30

    records = []
    frame_idx = 0
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        if frame_idx % process_every_n_frames == 0:
            frame_time_sec = frame_idx / fps
            lat, lon, accuracy_m = track.lookup(frame_time_sec)

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            result = predict(frame_rgb)

            if result["coverage_frac"] >= min_coverage_to_log:
                capture_time = (video_start_time + datetime.timedelta(seconds=frame_time_sec)).isoformat()
                record = build_issue_record(
                    result,
                    source={"type": "video_frame", "file": os.path.basename(video_path),
                            "frame_index": frame_idx, "frame_time_sec": round(frame_time_sec, 2)},
                    location={"latitude": lat, "longitude": lon, "source": "gps_log_interpolated",
                              "accuracy_m": accuracy_m, "timestamp": capture_time},
                )
                records.append(record)

        frame_idx += 1

    cap.release()
    return records


# ============================================================================
# Fallback: manual pin-drop for a clip with no GPS at all -- honestly labeled
# ============================================================================

def process_video_with_manual_pin(video_path: str, pin_lat: float, pin_lon: float,
                                   process_every_n_frames: int = 5, min_coverage_to_log: float = 0.005) -> list:
    print("NOTE: no real GPS for this clip -- using a single manual pin for the "
          "whole video. location.source is marked 'manual_pin_fallback' so this "
          "is never confused with real per-frame GPS in the output data.")
    cap = cv2.VideoCapture(video_path)
    records = []
    frame_idx = 0
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if frame_idx % process_every_n_frames == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            result = predict(frame_rgb)
            if result["coverage_frac"] >= min_coverage_to_log:
                record = build_issue_record(
                    result,
                    source={"type": "video_frame", "file": os.path.basename(video_path), "frame_index": frame_idx},
                    location={"latitude": pin_lat, "longitude": pin_lon,
                              "source": "manual_pin_fallback", "accuracy_m": None},
                )
                records.append(record)
        frame_idx += 1
    cap.release()
    return records


### Run it

Pick whichever matches what you have:
- Photos with location services on when they were taken -> `process_photos_with_exif`
- A video with a separately-recorded GPX track -> `process_video_with_gps_log`
- A video with no GPS at all -> `process_video_with_manual_pin` (same fallback used in the HUD demo above, now producing the same structured record schema)

In [ ]:
all_records = []

# --- Path A: photos with EXIF GPS ---
import glob
photo_paths = glob.glob("photos/*.jpg")   # <-- point at your photos
if photo_paths:
    all_records += process_photos_with_exif(photo_paths)

# --- Path B: video + GPX log (uncomment and fill in when you have a real log) ---
# all_records += process_video_with_gps_log(
#     video_path="survey_video.mp4",
#     gpx_path="gps_log.gpx",
#     video_start_time=datetime.datetime(2026, 8, 20, 9, 10, 0),  # when you pressed record
# )

# --- Fallback for clips with no GPS at all ---
# all_records += process_video_with_manual_pin(
#     video_path="some_clip.mp4", pin_lat=19.0760, pin_lon=72.8777,
# )

with open("issues_geotagged.json", "w") as f:
    json.dump(all_records, f, indent=2)

print(f"Wrote {len(all_records)} geotagged issue records to issues_geotagged.json")
if all_records:
    print(json.dumps(all_records[0], indent=2))  # peek at the schema


---
## 6. Test-set sanity check (optional)

Compares predictions against known ground-truth masks from the labeled test set. Useful for re-verifying the model against labeled data, separate from your own unlabeled real-world photos above. Needs the dataset present locally (see README for the download link) at `DATASET_DIR` below.

In [ ]:
import glob

DATASET_DIR = "data/Final Indian Footpath Damage Segmentation Dataset1"  # <-- set to your extracted dataset path
test_dir = os.path.join(DATASET_DIR, "Test")

if not os.path.isdir(test_dir):
    print(f"Dataset not found at {test_dir} -- skipping. See README for the dataset download link.")
else:
    test_jpgs = sorted(glob.glob(os.path.join(test_dir, "*.jpg")))[:3]
    print(f"Testing on {len(test_jpgs)} images from the held-out test set")

    for img_path in test_jpgs:
        name = os.path.splitext(os.path.basename(img_path))[0]
        gt_path = img_path[:-4] + ".png"
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        result = predict(image)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        resized_img = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        axes[0].imshow(resized_img); axes[0].set_title(f"{name} - Input"); axes[0].axis("off")
        if os.path.exists(gt_path):
            gt_mask = cv2.resize(cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE), (IMG_SIZE, IMG_SIZE))
            axes[1].imshow(gt_mask, cmap="gray"); axes[1].set_title("Ground truth")
        axes[1].axis("off")
        axes[2].imshow(resized_img); axes[2].imshow(result["raw_pred"], cmap="jet", alpha=0.45)
        axes[2].set_title(f"severity={result['severity']} ({result['severity_score']:.2f})")
        axes[2].axis("off")
        plt.tight_layout()
        plt.show()

        print(f"{name}: severity={result['severity']} ({result['severity_score']:.2f}) "
              f"coverage={result['coverage_frac']*100:.1f}%")


---
## Training and fine-tuning

Training-from-scratch and fine-tuning-on-hard-examples code has been moved out of this notebook into `train.py` and `finetune.py` in this repo, so that this notebook stays focused on "load the model, get predictions." See those files (and the README) if you want to reproduce or continue training. This repo ships `checkpoints/best_model.pt`, the checkpoint used throughout this notebook.
